<a href="https://colab.research.google.com/github/dindon04/whisper-speech-recognition/blob/main/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Installing libraries...")
!pip install -q transformers librosa soundfile evaluate jiwer datasets accelerate matplotlib pandas

import torch
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import re
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from evaluate import load

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Compute device: {device}")

wer_metric = load("wer")

print("Loading Whisper TINY...")
processor_t = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model_t = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)
model_t.config.forced_decoder_ids = None

print("Loading Whisper SMALL...")
processor_s = WhisperProcessor.from_pretrained("openai/whisper-small")
model_s = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)
model_s.config.forced_decoder_ids = None

print("\nLoading Google FLEURS dataset (ru_ru, 200 samples)...")
dataset = load_dataset("google/fleurs", "ru_ru", split="test", streaming=True)
dataset_subset = list(dataset.take(200))

In [ ]:
def add_noise(signal, snr_db, seed=None):
    if snr_db > 50: return signal
    if seed is not None:
        np.random.seed(seed)
    P_signal = np.mean(signal ** 2)
    P_noise = P_signal / (10 ** (snr_db/10))
    noise = np.random.normal(0, np.sqrt(P_noise), signal.shape)
    return signal + noise

def run_model(model, processor, audio_input):
    start_time = time.time()
    inputs = processor(audio_input, sampling_rate=16000, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        predicted_ids = model.generate(inputs)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].lower()
    proc_time = time.time() - start_time
    return transcription, proc_time

In [ ]:
snr_levels = [50, 25, 10, 5, 0, -5]
num_runs = 5
results = []

print("\nSTARTING STRESS TEST...")
print(f"{'SNR':<5} | {'Tiny WER':<10} | {'Small WER':<10} | {'RTF Tiny':<10} | {'RTF Small':<10}")
print("-" * 65)

for snr in snr_levels:
    wer_t_runs, wer_s_runs = [], []
    rtf_t_runs, rtf_s_runs = [], []

    for run_idx in range(num_runs):
        wer_t_files, wer_s_files = [], []
        rtf_t_files, rtf_s_files = [], []

        for idx, item in enumerate(dataset_subset):
            audio_array = item['audio']['array']
            orig_sr = item['audio']['sampling_rate']

            ref_text = item['transcription'].lower().strip()
            clean_ref = re.sub(r'[^\w\s]', '', ref_text)

            if orig_sr != 16000:
                audio_array = librosa.resample(audio_array, orig_sr=orig_sr, target_sr=16000)
            audio_duration = len(audio_array) / 16000

            noisy_audio = add_noise(audio_array, snr, seed=(run_idx * 1000 + idx))

            text_t, time_t = run_model(model_t, processor_t, noisy_audio)
            clean_t = re.sub(r'[^\w\s]', '', text_t)
            if not clean_t: clean_t = " "
            wer_t_files.append(wer_metric.compute(predictions=[clean_t], references=[clean_ref]))
            rtf_t_files.append(time_t / audio_duration)

            text_s, time_s = run_model(model_s, processor_s, noisy_audio)
            clean_s = re.sub(r'[^\w\s]', '', text_s)
            if not clean_s: clean_s = " "
            wer_s_files.append(wer_metric.compute(predictions=[clean_s], references=[clean_ref]))
            rtf_s_files.append(time_s / audio_duration)

        wer_t_runs.append(np.mean(wer_t_files))
        wer_s_runs.append(np.mean(wer_s_files))
        rtf_t_runs.append(np.mean(rtf_t_files))
        rtf_s_runs.append(np.mean(rtf_s_files))

    avg_wer_t = np.mean(wer_t_runs)
    avg_wer_s = np.mean(wer_s_runs)
    avg_rtf_t = np.mean(rtf_t_runs)
    avg_rtf_s = np.mean(rtf_s_runs)

    results.append({
        "SNR": snr,
        "WER_Tiny": avg_wer_t * 100,
        "WER_Small": avg_wer_s * 100,
        "RTF_Tiny": avg_rtf_t,
        "RTF_Small": avg_rtf_s
    })

    print(f"{snr:<5} | {avg_wer_t:<10.2%} | {avg_wer_s:<10.2%} | {avg_rtf_t:<10.4f} | {avg_rtf_s:<10.4f}")

In [ ]:
df = pd.DataFrame(results)
df.to_csv("diploma_experiment_results.csv", index=False)

plt.figure(figsize=(10,6))
plt.plot(df["SNR"], df["WER_Tiny"], 'b--o', label='Whisper Tiny (39M params)', linewidth=2)
plt.plot(df["SNR"], df["WER_Small"], 'r-o', label='Whisper Small (244M params)', linewidth=3)
plt.title('WER vs Noise Level (SNR) - Google FLEURS Dataset')
plt.xlabel('Signal-to-Noise Ratio (SNR), dB')
plt.ylabel('WER (%)')
plt.legend()
plt.grid(True)
plt.gca().invert_xaxis()
plt.savefig('fig_3_1_wer_comparison.png', dpi=300)
plt.show()

avg_rtf_t_total = df["RTF_Tiny"].mean()
avg_rtf_s_total = df["RTF_Small"].mean()

plt.figure(figsize=(7,5))
bars = plt.bar(['Whisper Tiny', 'Whisper Small'], [avg_rtf_t_total, avg_rtf_s_total], color=['blue', 'red'], alpha=0.7)
plt.title('Performance Comparison (RTF)')
plt.ylabel('RTF (lower is better)')
plt.grid(axis='y', linestyle='--')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval, round(yval, 4), ha='center', va='bottom')

plt.savefig('fig_3_2_rtf_comparison.png', dpi=300)
plt.show()

print("Done! All charts and tables saved.")